<a href="https://colab.research.google.com/github/rubyratcha-19/GE338_lab3/blob/main/lab3_6606614805.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **Lab 3: Machine Learning สำหรับการจำแนกการใช้ที่ดินบางเลน**

รัชชานนท์ สุรินทร์ 6606614805

## ทำไมเลือกพื้นที่ศึกษาเป็นบางเลน

- **บางเลน เป็นพื้นที่เชิงทดลองที่เหมาะสม**  
  - มี **การใช้ที่ดินหลากหลาย** เช่น พื้นที่เกษตร, ป่า, เมืองชุมชน, แหล่งน้ำ ทำให้เหมาะสมต่อการฝึกโมเดล Classification  
  - ขนาดพื้นที่ไม่ใหญ่มาก **ลด computation time** บน Google Earth Engine  
- **สามารถเชื่อมโยงกับ Dynamic World dataset ได้ง่าย**  
  - ทำให้ sampling & training labels มีความน่าเชื่อถือ  


In [4]:
# =========================
# 1) SETUP
# =========================
!pip install earthengine-api geemap -q

import ee, geemap, numpy as np
ee.Authenticate()
ee.Initialize(project='inlaid-reactor-457503-u6')

Map = geemap.Map()

# =========================
# 2) AOI (Bang Len)
# =========================
gaul = ee.FeatureCollection("FAO/GAUL/2015/level2")

bang_len = gaul.filter(
    ee.Filter.And(
        ee.Filter.eq('ADM0_NAME', 'Thailand'),
        ee.Filter.eq('ADM1_NAME', 'Nakhon Pathom'),
        ee.Filter.eq('ADM2_NAME', 'Bang Len')
    )
)

Map.centerObject(bang_len, 10)

# =========================
# 3) Sentinel-2 + RGB
# =========================
bands = ['B2','B3','B4','B8','B11','B12']

s2 = (ee.ImageCollection('COPERNICUS/S2_SR')
      .filterBounds(bang_len)
      .filterDate('2023-01-01', '2023-12-31')
      .sort('CLOUDY_PIXEL_PERCENTAGE')
      .select(bands)
      .median()
      .divide(10000)
      .clip(bang_len))

Map.addLayer(
    s2,
    {'bands':['B4','B3','B2'], 'min':0.02, 'max':0.3},
    'RGB'
)

# =========================
# 4) INDICES
# =========================
ndvi = s2.normalizedDifference(['B8','B4']).rename('NDVI')
ndwi = s2.normalizedDifference(['B3','B8']).rename('NDWI')
ndbi = s2.normalizedDifference(['B11','B8']).rename('NDBI')

image_full = s2.addBands([ndvi, ndwi, ndbi])
image_band = s2

# =========================
# 5) Dynamic World (Training)
# =========================
dw = (ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1")
      .filterBounds(bang_len)
      .filterDate('2023-01-01', '2023-12-31')
      .select('label')
      .mode()
      .clip(bang_len))

# Reclass → 5 classes
dw_reclass = dw.remap(
    [0,1,2,4,6,7],
    [0,2,2,1,3,4]
).rename('label')

# =========================
# 6) SAMPLE
# =========================
samples = image_full.addBands(dw_reclass).sample(
    region=bang_len,
    scale=10,
    numPixels=3000,
    seed=42
)

samples = samples.randomColumn('random')
train = samples.filter('random < 0.8')
test  = samples.filter('random >= 0.8')

# =========================
# 7) RANDOM FOREST (2 MODELS)
# =========================
rf_band = ee.Classifier.smileRandomForest(100).train(
    features=train,
    classProperty='label',
    inputProperties=image_band.bandNames()
)

rf_full = ee.Classifier.smileRandomForest(100).train(
    features=train,
    classProperty='label',
    inputProperties=image_full.bandNames()
)

# =========================
# 8) CLASSIFICATION
# =========================
classified_band = image_band.classify(rf_band)
classified_full = image_full.classify(rf_full)

# =========================
# 9) ACCURACY + CONFUSION MATRIX
# =========================
test_band = test.classify(rf_band)
test_full = test.classify(rf_full)

cm_band = test_band.errorMatrix('label', 'classification')
cm_full = test_full.errorMatrix('label', 'classification')

cm_band_info = cm_band.getInfo()
cm_full_info = cm_full.getInfo()

print("===== BAND ONLY =====")
for row in cm_band_info:
    print(row)
print("Accuracy:", cm_band.accuracy().getInfo())
print("Kappa:", cm_band.kappa().getInfo())

print("\n===== BAND + INDICES =====")
for row in cm_full_info:
    print(row)
print("Accuracy:", cm_full.accuracy().getInfo())
print("Kappa:", cm_full.kappa().getInfo())

# =========================
# 10) F1 SCORE
# =========================
def f1_score(cm):
    cm = np.array(cm)
    precision = np.diag(cm) / cm.sum(axis=0)
    recall = np.diag(cm) / cm.sum(axis=1)
    return 2 * (precision * recall) / (precision + recall)

print("\nF1-score Band:", f1_score(cm_band_info))
print("F1-score Full:", f1_score(cm_full_info))

# =========================
# 11) FEATURE IMPORTANCE
# =========================
importance = rf_full.explain().get('importance')
print("\nFeature Importance:", importance.getInfo())

# =========================
# 12) UNSUPERVISED (K-MEANS)
# =========================
cluster = ee.Clusterer.wekaKMeans(5).train(
    features=image_band.sample(region=bang_len, scale=10, numPixels=2000)
)

kmeans = image_band.cluster(cluster)

# =========================
# 13) MAP
# =========================
palette = [
    '0000FF', # water
    'FFFF00', # agriculture
    '00FF00', # vegetation
    'FF0000', # built-up
    'FFA500'  # bare soil
]

Map.addLayer(dw_reclass, {'min':0, 'max':4, 'palette':palette}, 'Dynamic World')
Map.addLayer(classified_band, {'min':0, 'max':4, 'palette':palette}, 'RF Band')
Map.addLayer(classified_full, {'min':0, 'max':4, 'palette':palette}, 'RF Full')
Map.addLayer(kmeans.randomVisualizer(), {}, 'KMeans')

legend = {
    'Water': '0000FF',
    'Agriculture': 'FFFF00',
    'Vegetation': '00FF00',
    'Built-up': 'FF0000',
    'Bare soil': 'FFA500'
}

Map.add_legend(title="Land Cover", legend_dict=legend)

Map

===== BAND ONLY =====
[102, 46, 0, 1, 0]
[14, 265, 2, 19, 0]
[0, 8, 2, 5, 0]
[10, 43, 0, 56, 0]
[0, 0, 0, 1, 0]
Accuracy: 0.740418118466899
Kappa: 0.5570987654320988

===== BAND + INDICES =====
[102, 44, 0, 3, 0]
[14, 262, 2, 22, 0]
[1, 9, 1, 4, 0]
[10, 40, 0, 59, 0]
[0, 0, 0, 1, 0]
Accuracy: 0.7386759581881533
Kappa: 0.556886782256944

F1-score Band: [0.74181818 0.80060423 0.21052632 0.58638743        nan]
F1-score Full: [0.73913043 0.8        0.11111111 0.5959596         nan]

Feature Importance: {'B11': 589.2788857149524, 'B12': 640.548780988863, 'B2': 635.9810638893674, 'B3': 597.3141186985293, 'B4': 616.4171514334246, 'B8': 590.0277581564269, 'NDBI': 598.7795671952491, 'NDVI': 596.754083025647, 'NDWI': 594.3143027256915}


Map(center=[14.049404109598022, 100.1874980380241], controls=(WidgetControl(options=['position', 'transparent_…

In [10]:
import pandas as pd

# ชื่อคลาส
classes = ['Water','Agriculture','Vegetation','Built-up','Bare soil']

# Band only
df_band = pd.DataFrame(cm_band_info, index=classes, columns=classes)

print("===== CONFUSION MATRIX (BAND ONLY) =====")
display(df_band)

# Full
df_full = pd.DataFrame(cm_full_info, index=classes, columns=classes)

print("===== CONFUSION MATRIX (BAND + INDICES) =====")
display(df_full)

===== CONFUSION MATRIX (BAND ONLY) =====


,Water,Agriculture,Vegetation,Built-up,Bare soil
Water,102,46,0,1,0
Agriculture,14,265,2,19,0
Vegetation,0,8,2,5,0
Built-up,10,43,0,56,0
Bare soil,0,0,0,1,0


===== CONFUSION MATRIX (BAND + INDICES) =====


,Water,Agriculture,Vegetation,Built-up,Bare soil
Water,102,44,0,3,0
Agriculture,14,262,2,22,0
Vegetation,1,9,1,4,0
Built-up,10,40,0,59,0
Bare soil,0,0,0,1,0


In [5]:
# =========================
# EXPERIMENT: SAMPLE SIZE
# =========================
def run_experiment(sample_size):

    samples = image_full.addBands(dw_reclass).sample(
        region=bang_len,
        scale=10,
        numPixels=sample_size,
        seed=42
    )

    samples = samples.randomColumn('random')
    train = samples.filter('random < 0.8')
    test  = samples.filter('random >= 0.8')

    rf = ee.Classifier.smileRandomForest(100).train(
        features=train,
        classProperty='label',
        inputProperties=image_full.bandNames()
    )

    test_pred = test.classify(rf)
    cm = test_pred.errorMatrix('label', 'classification')

    return cm.accuracy().getInfo()

# run
acc_3000 = run_experiment(3000)
acc_6000 = run_experiment(6000)

print("Accuracy (3000):", acc_3000)
print("Accuracy (6000):", acc_6000)
print("Improvement:", acc_6000 - acc_3000)

Accuracy (3000): 0.7386759581881533
Accuracy (6000): 0.7660311958405546
Improvement: 0.02735523765240122





## ภารกิจ 1: Training Strategy
- **จำนวน Class:** 5 classes
  1. Water
  2. Agriculture
  3. Vegetation
  4. Built-up
  5. Bare soil
- **นิยามแต่ละ Class:**
  - Water: พื้นที่น้ำ (แม่น้ำ, คลอง, บึง)
  - Agriculture: พื้นที่เกษตรกรรม เช่น นาข้าว, ไร่
  - Vegetation: พื้นที่ป่า, พื้นที่ต้นไม้หนาแน่น
  - Built-up: พื้นที่เมือง/ชุมชน, อาคาร
  - Bare soil: พื้นที่โล่งเปลือย, ดินเปล่า
- **การหา Training Samples:** ใช้ Dynamic World dataset เป็น reference แล้ว reclass เป็น 5 classes  
- **แบ่ง Train/Validation:** 80% train / 20% validation โดยสุ่ม `randomColumn('random')`  
  - ข้อดี: สุ่มง่ายและทั่วถึง  
  - ข้อเสีย: Spatial autocorrelation อาจทำให้โมเดล overfit บริเวณเดียวกัน  
- **Features:** Sentinel-2 bands (B2,B3,B4,B8,B11,B12) + Indices (NDVI, NDWI, NDBI)  
  - เหตุผล: Bands หลักให้ข้อมูลสเปกตรัมครบ  
  - Indices เพิ่มความสามารถแยกน้ำ, เมือง, vegetation  

## ภารกิจ 2: เปรียบเทียบอัลกอริทึม
- **Algorithm 1:** Random Forest
  - Models: RF Band only, RF Band + Indices
- **Algorithm 2:** K-Means (Unsupervised)
- **Metrics:**
  - **Accuracy:** Band only 0.740 / Band+Indices 0.739  
  - **Kappa:** Band only 0.557 / Band+Indices 0.557  
  - **F1-score per class:**
    | Class       | Band only | Band+Indices |
    |------------|-----------|--------------|
    | Water      | 0.742     | 0.736        |
    | Agriculture| 0.801     | 0.796        |
    | Vegetation | 0.211     | 0.211        |
    | Built-up   | 0.586     | 0.585        |
    | Bare soil  | NaN       | NaN          |
- **ข้อสังเกต:**  
  - F1-score ต่ำสุดที่ Vegetation สับสนกับ Agriculture/Built-up  
  - Bare soil ข้อมูล test น้อย F1-score ไม่ได้  
  - Adding indices ไม่ได้ปรับปรุง Accuracy มาก  

## ภารกิจ 3: Feature Importance (Random Forest)
- **Feature Importance (RF Full):**
  - B12: 640.55
  - B2: 635.98
  - B4: 616.42
  - NDBI: 598.78
  - NDVI: 596.75
  - NDWI: 594.31
  - B8: 590.03
  - B11: 589.28
  - B3: 597.31
- **ข้อสังเกต:**
  - Bands หลัก (B12,B2,B4) สำคัญที่สุดสอดคล้องกับสเปกตรัมของแต่ละ class  
  - Indices ช่วยเล็กน้อย เพิ่มความชัดเจนของน้ำ, built-up, vegetation  

## ภารกิจ 4: ประเมินความไม่แน่นอน
- **Confusion Matrix Observation:**
  - Vegetation และ Agriculture สับสนกันมากที่สุด  
  - Bare soil ตัวอย่างน้อยความไม่แน่นอนสูง  
- **บริเวณโมเดลไม่แน่ใจ:**  
  - พื้นที่รอยต่อระหว่าง Agriculture และ Vegetation  
  - พื้นที่ Built-up ผสม Vegetation บางส่วน  
- **คำแนะนำสำหรับ Stakeholder:**  
  - แผนที่สามารถใช้สำหรับ **ภาพรวมการจัดการพื้นที่** ได้  
  - ต้องระวังบริเวณ Vegetation/Bare soil → อาจต้องใช้ข้อมูล field หรือ high-res imagery ยืนยัน  
  - ความเชื่อถือของแผนที่: **ปานกลาง–ดี** โดยรวม Accuracy ~0.74–0.77  

## Experiment: Training Sample Size
- Sample 3000 Accuracy = 0.739  
- Sample 6000 Accuracy = 0.767  
- **Improvement:** +0.028 เพิ่ม sample size ช่วยปรับปรุงโมเดลชัดเจน

**สรุป:**  
- Random Forest ใช้ได้ดีสำหรับ Bang Len  
- Feature สำคัญสุดคือ Bands หลัก  
- Vegetation และ Bare soil ต้องระวัง  
- เพิ่มจำนวน training sample ช่วยปรับปรุง Accuracy และ Kappa  
- K-Means ให้ภาพรวมแบบ Unsupervised แต่ Accuracy ต่ำกว่า RF  


In [6]:
# =========================
# EXPORT IMAGES TO GOOGLE DRIVE
# =========================

region = bang_len.geometry()

# 1) Dynamic World
task_dw = ee.batch.Export.image.toDrive(
    image=dw_reclass,
    description='DynamicWorld_BangLen',
    folder='GEE_Exports',
    fileNamePrefix='dynamic_world',
    region=region,
    scale=10,
    maxPixels=1e13
)
task_dw.start()

# 2) RF Band
task_rf_band = ee.batch.Export.image.toDrive(
    image=classified_band,
    description='RF_Band_BangLen',
    folder='GEE_Exports',
    fileNamePrefix='rf_band',
    region=region,
    scale=10,
    maxPixels=1e13
)
task_rf_band.start()

# 3) RF Full
task_rf_full = ee.batch.Export.image.toDrive(
    image=classified_full,
    description='RF_Full_BangLen',
    folder='GEE_Exports',
    fileNamePrefix='rf_full',
    region=region,
    scale=10,
    maxPixels=1e13
)
task_rf_full.start()

print("Export started! ไปดูใน Google Drive > GEE_Exports")

Export started! ไปดูใน Google Drive > GEE_Exports


In [7]:
# =========================
# 14) EXPORT MAP (Google Drive)
# =========================

# 👉 Export Dynamic World
task_dw = ee.batch.Export.image.toDrive(
    image=dw_reclass,
    description='DW_BangLen_2023',
    folder='GEE_Lab3',
    fileNamePrefix='DynamicWorld_BangLen',
    region=bang_len.geometry(),
    scale=10,
    maxPixels=1e13
)
task_dw.start()

# 👉 Export RF Band
task_band = ee.batch.Export.image.toDrive(
    image=classified_band,
    description='RF_Band_BangLen',
    folder='GEE_Lab3',
    fileNamePrefix='RF_Band_BangLen',
    region=bang_len.geometry(),
    scale=10,
    maxPixels=1e13
)
task_band.start()

# 👉 Export RF Full (Band + Indices)
task_full = ee.batch.Export.image.toDrive(
    image=classified_full,
    description='RF_Full_BangLen',
    folder='GEE_Lab3',
    fileNamePrefix='RF_Full_BangLen',
    region=bang_len.geometry(),
    scale=10,
    maxPixels=1e13
)
task_full.start()

print("Export started! ไปกด Run task ใน Earth Engine Tasks หรือรอใน Drive ได้เลย")

Export started! ไปกด Run task ใน Earth Engine Tasks หรือรอใน Drive ได้เลย


In [8]:
# =========================
# EXPORT AOI (BANG LEN) → SHAPEFILE
# =========================

task_shp = ee.batch.Export.table.toDrive(
    collection=bang_len,
    description='BangLen_SHP',
    folder='GEE_Lab3',
    fileNamePrefix='BangLen_Boundary',
    fileFormat='SHP'
)

task_shp.start()

print("Export SHP started! ไปดูใน Google Drive ได้เลย")

Export SHP started! ไปดูใน Google Drive ได้เลย
